# Partie 1 : Exploration SQL et Profilage du Churn
L'objectif de cette section est d'interroger notre base de données SQLite pour comprendre la répartition globale du churn et identifier les premiers "Red Flags" (signaux d'alarme) chez nos clients.

### 1. Quel est le taux de résiliation global de l'entreprise ?
Avant toute chose, nous devons connaître la proportion exacte de clients qui quittent l'entreprise pour établir notre "Baseline" (notre point de référence).

In [1]:
import sqlite3
import pandas as pd

# Connexion à la base de données (on remonte d'un dossier puis on va dans data)
conn = sqlite3.connect('../data/telco_churn.db')

# Requête 1 : Le Taux de Churn Global
query_global_churn = """
SELECT 
    Churn as Resiliation,
    COUNT(*) as Nombre_Clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 1) as Pourcentage_Total
FROM customers
GROUP BY Churn;
"""

df_churn = pd.read_sql_query(query_global_churn, conn)
display(df_churn)

,Resiliation,Nombre_Clients,Pourcentage_Total
0,No,5174,73.5
1,Yes,1869,26.5


**Conclusion :** Près de **26,5 %** de nos clients ont résilié leur abonnement. C'est un taux d'attrition très élevé pour une entreprise de télécommunications, ce qui justifie pleinement le besoin de mettre en place un algorithme prédictif pour retenir ces clients.

---
### 2. Le type de contrat influence-t-il le départ des clients ?
Logiquement, un client sans engagement est plus susceptible de partir. Vérifions cette hypothèse avec les données.

In [2]:
# Requête 2 : Churn par type de contrat
query_contract = """
SELECT 
    Contract as Type_Contrat,
    COUNT(*) as Nombre_Total,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as Nombre_Departs,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as Taux_de_Churn_Pourcent
FROM customers
GROUP BY Contract
ORDER BY Taux_de_Churn_Pourcent DESC;
"""

df_contract = pd.read_sql_query(query_contract, conn)
display(df_contract)

,Type_Contrat,Nombre_Total,Nombre_Departs,Taux_de_Churn_Pourcent
0,Month-to-month,3875,1655,42.7
1,One year,1473,166,11.3
2,Two year,1695,48,2.8


**Conclusion :** L'hypothèse est confirmée et l'écart est massif. Plus de **42 %** des clients en contrat mensuel ("Month-to-month") finissent par partir, contre seulement **2,8 %** pour les contrats de deux ans. Le type de contrat sera une variable prédictive majeure pour notre futur modèle de Machine Learning.

---
### 3. Les clients qui partent paient-ils plus cher ?
Analysons si la tarification (MonthlyCharges) et l'ancienneté (tenure) jouent un rôle dans la décision de résilier.

In [3]:
# Requête 3 : Ancienneté et Prix moyen selon le Churn
query_revenue = """
SELECT 
    Churn,
    COUNT(*) as Nombre_Clients,
    ROUND(AVG(tenure), 1) as Anciennete_Moyenne_Mois,
    ROUND(AVG(MonthlyCharges), 2) as Facture_Mensuelle_Moyenne
FROM customers
GROUP BY Churn;
"""

df_revenue = pd.read_sql_query(query_revenue, conn)
display(df_revenue)

# Fermeture de la connexion à la base
conn.close()

,Churn,Nombre_Clients,Anciennete_Moyenne_Mois,Facture_Mensuelle_Moyenne
0,No,5174,37.6,61.27
1,Yes,1869,18.0,74.44


**Conclusion :** Le profil type du client qui résilie se dessine clairement. Il reste beaucoup moins longtemps (18 mois en moyenne contre 37 mois pour un client fidèle) et **paie une facture mensuelle nettement plus élevée** (74,44 contre 61,26). Il y a probablement un problème de rapport qualité/prix sur certains services additionnels.